In [1]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout
)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [3]:
df = pd.read_csv("../data/ndvi_dataset.csv")

print("Dataset shape:", df.shape)

Dataset shape: (908, 27)


In [6]:
ndvi_cols = [
    col for col in df.columns
    if col.startswith("ndvi_")
]

print("Number of NDVI columns:", len(ndvi_cols))
print(ndvi_cols)


Number of NDVI columns: 24
['ndvi_00', 'ndvi_01', 'ndvi_02', 'ndvi_03', 'ndvi_04', 'ndvi_05', 'ndvi_06', 'ndvi_07', 'ndvi_08', 'ndvi_09', 'ndvi_10', 'ndvi_11', 'ndvi_12', 'ndvi_13', 'ndvi_14', 'ndvi_15', 'ndvi_16', 'ndvi_17', 'ndvi_18', 'ndvi_19', 'ndvi_20', 'ndvi_21', 'ndvi_22', 'ndvi_23']


In [7]:
X = df[ndvi_cols].values
y = df["label"].values

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (908, 24)
y shape: (908,)


In [8]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Classes:", label_encoder.classes_)
print("Encoded labels:", np.unique(y_encoded))

Classes: ['deforested' 'forest' 'old_clearing']
Encoded labels: [0 1 2]


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (726, 24)
X_test shape: (182, 24)
y_train shape: (726,)
y_test shape: (182,)


In [10]:
X_train = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)

X_test = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (726, 24, 1)
X_test shape: (182, 24, 1)


In [11]:
cnn_model = Sequential([
    
    Conv1D(
        filters=32,
        kernel_size=3,
        activation="relu",
        input_shape=(24, 1)
    ),

    MaxPooling1D(
        pool_size=2
    ),

    Conv1D(
        filters=64,
        kernel_size=3,
        activation="relu"
    ),

    MaxPooling1D(
        pool_size=2
    ),

    Flatten(),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.3),

    Dense(
        3,
        activation="softmax"
    )
])

cnn_model.summary()

/opt/anaconda3/envs/ndvi-cnn/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 22, 32)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 11, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 9, 64)          │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 4, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,979 (89.76 KB)

 Trainable params: 22,979 (89.76 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("CNN compiled successfully.")

CNN compiled successfully.


In [13]:
history = cnn_model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=30,
    batch_size=32,
    verbose=1
)

Epoch 1/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5431 - loss: 1.0155 - val_accuracy: 0.5068 - val_loss: 0.9896
Epoch 2/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5621 - loss: 0.9443 - val_accuracy: 0.5068 - val_loss: 0.9916
Epoch 3/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5621 - loss: 0.9434 - val_accuracy: 0.5068 - val_loss: 0.9818
Epoch 4/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5603 - loss: 0.9382 - val_accuracy: 0.5068 - val_loss: 0.9823
Epoch 5/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5621 - loss: 0.9369 - val_accuracy: 0.5068 - val_loss: 0.9791
Epoch 6/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5621 - loss: 0.9305 - val_accuracy: 0.5068 - val_loss: 0.9776
Epoch 7/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5655 - loss: 0.9271 - val_accuracy: 0.5068 - val_loss: 0.9728
Epoch 8/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5621 - loss: 0.9277 - val_accuracy: 0.5068 - val_loss:

In [14]:
test_loss, test_accuracy = cnn_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("CNN Test Loss:", test_loss)
print("CNN Test Accuracy:", test_accuracy)

CNN Test Loss: 0.7977034449577332
CNN Test Accuracy: 0.692307710647583


In [15]:
y_prob = cnn_model.predict(X_test, verbose=0)

y_pred = np.argmax(y_prob, axis=1)

print("CNN Accuracy:",
      accuracy_score(y_test, y_pred))

print("CNN Macro Precision:",
      precision_score(y_test, y_pred, average="macro"))

print("CNN Macro Recall:",
      recall_score(y_test, y_pred, average="macro"))

print("CNN Macro F1:",
      f1_score(y_test, y_pred, average="macro"))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)

CNN Accuracy: 0.6923076923076923
CNN Macro Precision: 0.6313932980599647
CNN Macro Recall: 0.5191820191599116
CNN Macro F1: 0.5074926253687315

Classification Report:
              precision    recall  f1-score   support

  deforested       0.50      0.04      0.08        23
      forest       0.69      0.87      0.77       100
old_clearing       0.70      0.64      0.67        59

    accuracy                           0.69       182
   macro avg       0.63      0.52      0.51       182
weighted avg       0.67      0.69      0.65       182



In [16]:
cm_cnn = confusion_matrix(y_test, y_pred)

print("CNN Confusion Matrix:")
print(
    pd.DataFrame(
        cm_cnn,
        index=[f"Actual {c}" for c in label_encoder.classes_],
        columns=[f"Pred {c}" for c in label_encoder.classes_]
    )
)

CNN Confusion Matrix:
                     Pred deforested  Pred forest  Pred old_clearing
Actual deforested                  1           19                  3
Actual forest                      0           87                 13
Actual old_clearing                1           20                 38


In [17]:
cnn_results = {
    "Model": "CNN",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Macro Precision": precision_score(
        y_test, y_pred, average="macro"
    ),
    "Macro Recall": recall_score(
        y_test, y_pred, average="macro"
    ),
    "Macro F1": f1_score(
        y_test, y_pred, average="macro"
    )
}

print(cnn_results)

{'Model': 'CNN', 'Accuracy': 0.6923076923076923, 'Macro Precision': 0.6313932980599647, 'Macro Recall': 0.5191820191599116, 'Macro F1': 0.5074926253687315}
